In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="FR",
    max_iter=10000,
    tol=1e-100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.23592589795589447
epoch:  1, loss: 0.1451844722032547
epoch:  2, loss: 0.04688439518213272
epoch:  3, loss: 0.042375821620225906
epoch:  4, loss: 0.03655453771352768
epoch:  5, loss: 0.036459799855947495
epoch:  6, loss: 0.0363280326128006
epoch:  7, loss: 0.036261096596717834
epoch:  8, loss: 0.03625275194644928
epoch:  9, loss: 0.03623320534825325
epoch:  10, loss: 0.03620780259370804
epoch:  11, loss: 0.036130957305431366
epoch:  12, loss: 0.03612244501709938
epoch:  13, loss: 0.035947564989328384
epoch:  14, loss: 0.035947564989328384
epoch:  15, loss: 0.03568904101848602
epoch:  16, loss: 0.03567764163017273
epoch:  17, loss: 0.035079196095466614
epoch:  18, loss: 0.035079196095466614
epoch:  19, loss: 0.034535039216279984
epoch:  20, loss: 0.034535039216279984
epoch:  21, loss: 0.0340186208486557
epoch:  22, loss: 0.0340186208486557
epoch:  23, loss: 0.03349049389362335
epoch:  24, loss: 0.03349049389362335
epoch:  25, loss: 0.03288736939430237
epoch:  26, loss

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6842378318187516
Test metrics:  R2 = 0.7004402529230499


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.33542704582214355
epoch:  1, loss: 0.19591392576694489
epoch:  2, loss: 0.10592932999134064
epoch:  3, loss: 0.06141609698534012
epoch:  4, loss: 0.04369661211967468
epoch:  5, loss: 0.038023486733436584
epoch:  6, loss: 0.036608073860406876
epoch:  7, loss: 0.036346081644296646
epoch:  8, loss: 0.0363105908036232
epoch:  9, loss: 0.036304064095020294
epoch:  10, loss: 0.03626395761966705
epoch:  11, loss: 0.03624175488948822
epoch:  12, loss: 0.03623582795262337
epoch:  13, loss: 0.03621627762913704
epoch:  14, loss: 0.036204319447278976
epoch:  15, loss: 0.036200329661369324
epoch:  16, loss: 0.03615304455161095
epoch:  17, loss: 0.03610673174262047
epoch:  18, loss: 0.03606458753347397
epoch:  19, loss: 0.03606007248163223
epoch:  20, loss: 0.03601423650979996
epoch:  21, loss: 0.03600949048995972
epoch:  22, loss: 0.0359831228852272
epoch:  23, loss: 0.035969194024801254
epoch:  24, loss: 0.03596382215619087
epoch:  25, loss: 0.03590964898467064
epoch:  26, loss:

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7013986823703641
Test metrics:  R2 = 0.7163567140019851


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2838250994682312
epoch:  1, loss: 0.15634387731552124
epoch:  2, loss: 0.09767694771289825
epoch:  3, loss: 0.06859349459409714
epoch:  4, loss: 0.05353081598877907
epoch:  5, loss: 0.04558075591921806
epoch:  6, loss: 0.04134215787053108
epoch:  7, loss: 0.03907133638858795
epoch:  8, loss: 0.037851959466934204
epoch:  9, loss: 0.037195079028606415
epoch:  10, loss: 0.03684026375412941
epoch:  11, loss: 0.036648012697696686
epoch:  12, loss: 0.03654282167553902
epoch:  13, loss: 0.036484669893980026
epoch:  14, loss: 0.036451857537031174
epoch:  15, loss: 0.03643251955509186
epoch:  16, loss: 0.03643136844038963
epoch:  17, loss: 0.03639063611626625
epoch:  18, loss: 0.03636176884174347
epoch:  19, loss: 0.036357708275318146
epoch:  20, loss: 0.0363239161670208
epoch:  21, loss: 0.03632016107439995
epoch:  22, loss: 0.03624827787280083
epoch:  23, loss: 0.036243777722120285
epoch:  24, loss: 0.0360223688185215
epoch:  25, loss: 0.03593094274401665
epoch:  26, loss: 

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7150256524822145
Test metrics:  R2 = 0.7258519349407861


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="HS"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.17109328508377075
epoch:  1, loss: 0.10898279398679733
epoch:  2, loss: 0.053759828209877014
epoch:  3, loss: 0.040964096784591675
epoch:  4, loss: 0.03807385638356209
epoch:  5, loss: 0.03700695559382439
epoch:  6, loss: 0.03663612902164459
epoch:  7, loss: 0.03647451475262642
epoch:  8, loss: 0.03623197600245476
epoch:  9, loss: 0.03604825586080551
epoch:  10, loss: 0.035960230976343155
epoch:  11, loss: 0.035926204174757004
epoch:  12, loss: 0.03591059893369675
epoch:  13, loss: 0.035871006548404694
epoch:  14, loss: 0.035854849964380264
epoch:  15, loss: 0.03584472835063934
epoch:  16, loss: 0.035836029797792435
epoch:  17, loss: 0.03581065684556961
epoch:  18, loss: 0.03580004349350929
epoch:  19, loss: 0.03579045087099075
epoch:  20, loss: 0.03576458618044853
epoch:  21, loss: 0.03575359657406807
epoch:  22, loss: 0.03574462980031967
epoch:  23, loss: 0.03571619093418121
epoch:  24, loss: 0.03570554777979851
epoch:  25, loss: 0.035685814917087555
epoch:  26, lo

In [12]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6993186365145427
Test metrics:  R2 = 0.7368538187918123


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="DY"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.43499746918678284
epoch:  1, loss: 0.2539007067680359
epoch:  2, loss: 0.03680713474750519
epoch:  3, loss: 0.03668644651770592
epoch:  4, loss: 0.036563441157341
epoch:  5, loss: 0.036560241132974625
epoch:  6, loss: 0.03654869273304939
epoch:  7, loss: 0.03647072613239288
epoch:  8, loss: 0.036154694855213165
epoch:  9, loss: 0.03604956716299057
epoch:  10, loss: 0.03602374345064163
epoch:  11, loss: 0.03585486859083176
epoch:  12, loss: 0.03552158921957016
epoch:  13, loss: 0.03517615422606468
epoch:  14, loss: 0.03507497161626816
epoch:  15, loss: 0.034966036677360535
epoch:  16, loss: 0.0348849818110466
epoch:  17, loss: 0.03485741466283798
epoch:  18, loss: 0.03484202176332474
epoch:  19, loss: 0.034810181707143784
epoch:  20, loss: 0.03478410094976425
epoch:  21, loss: 0.03475392982363701
epoch:  22, loss: 0.034739214926958084
epoch:  23, loss: 0.03470183536410332
epoch:  24, loss: 0.03468594327569008
epoch:  25, loss: 0.034637659788131714
epoch:  26, loss: 0.

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6475757522759452
Test metrics:  R2 = 0.6737557483207008
